# TF-IDF Feature Extraction

## What this notebook does

Converts `processed_text` (space-joined lemmatized tokens from notebook 04) into a sparse TF-IDF matrix ready for classification.

## Why split before vectorizing

TF-IDF learns vocabulary and IDF weights from the data it is fitted on. Fitting on all 8,077 rows and then splitting means test tickets influenced vocabulary construction — that is data leakage. The vectorizer must be **fitted only on training data**, then **applied to test data without refitting**.

## Outputs

| File | Format | Used by |
|---|---|---|
| `models/tfidf_vectorizer.pkl` | joblib pickle | Notebook 08 — classification, Streamlit app inference |
| `data/processed/X_train_tfidf.npz` | scipy sparse | Notebook 08 |
| `data/processed/X_test_tfidf.npz` | scipy sparse | Notebook 08 |
| `data/processed/y_train_type.csv` | CSV | Notebook 08 |
| `data/processed/y_test_type.csv` | CSV | Notebook 08 / 09 |
| `data/processed/y_train_priority.csv` | CSV | Notebook 10 |
| `data/processed/y_test_priority.csv` | CSV | Notebook 10 |

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

PREPROCESSED_PATH = '../data/processed/preprocessed_tasks.csv'
MODELS_DIR        = '../models/'
PROCESSED_DIR     = '../data/processed/'

os.makedirs(MODELS_DIR,    exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Step 0 — Verify preprocessed_tasks.csv

Three checks before touching sklearn:

1. **No nulls AND no empty strings in `processed_text`** — `isnull()` silently misses empty strings `""`. After detemplatization, some tickets got shorter; a few may have become `""` rather than `NaN`. Check both.
2. **Word count distribution** — print min, mean, max. Expected mean ≈ 20.8 from notebook 04. CSV round-trips can introduce subtle encoding issues — this confirms the data survived intact.
3. **Label class distributions** — 5 balanced ticket types, 4 balanced priority levels. Must match notebook 04 output exactly.

In [ ]:
df = pd.read_csv(PREPROCESSED_PATH)
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}\n")

# Check 1 — nulls and empty strings
print("=== CHECK 1: NULL AND EMPTY STRING SCAN ===\n")
null_count  = df['processed_text'].isnull().sum()
empty_count = (df['processed_text'].str.strip() == '').sum()
print(f"NaN values in processed_text   : {null_count}")
print(f"Empty strings in processed_text: {empty_count}")
if null_count == 0 and empty_count == 0:
    print("\n✓ Both zero — safe to proceed")
else:
    print("\n✗ ACTION REQUIRED — inspect or drop these rows before fitting TF-IDF")
    problem_rows = df[df['processed_text'].isnull() | (df['processed_text'].str.strip() == '')]
    print(problem_rows[['processed_text', 'task_type', 'priority']].head(10))

# Check 2 — word count distribution
print("\n=== CHECK 2: WORD COUNT DISTRIBUTION ===\n")
wc = df['processed_text'].apply(lambda x: len(x.split()) if isinstance(x, str) and x.strip() else 0)
print(f"Min  : {wc.min()}")
print(f"Mean : {wc.mean():.1f}   (expected ≈ 20.8 from notebook 04)")
print(f"Max  : {wc.max()}")

# Check 3 — label distributions
print("\n=== CHECK 3: LABEL CLASS DISTRIBUTIONS ===\n")
print("task_type value counts:")
print(df['task_type'].value_counts().to_string())
print("\npriority value counts:")
print(df['priority'].value_counts().to_string())

## Step 1 — Train-Test Split

**Split before fitting** — the vectorizer sees only training data.

**Why stratify:** 5 classes range from 1,571 to 1,659 tickets. Without stratification, class proportions in the test set could deviate by chance. `stratify=y_type` guarantees each class appears in the test set in proportion to its share of the full dataset.

**Label alignment:** `y_priority` must use the **same row indices** as `X_train` / `X_test`. The correct pattern is `.loc[X_train.index]` — this reuses the exact rows from the first split rather than running a second independent split, which could silently misalign label arrays.

| Split | Rows |
|---|---|
| Training | 6,461 (80%) |
| Test | 1,616 (20%) |

In [ ]:
X          = df['processed_text']
y_type     = df['task_type']
y_priority = df['priority']

X_train, X_test, y_train_type, y_test_type = train_test_split(
    X, y_type,
    test_size=0.2,
    random_state=42,
    stratify=y_type
)

# Reuse split indices — guarantees row alignment across both label arrays
y_train_priority = y_priority.loc[X_train.index]
y_test_priority  = y_priority.loc[X_test.index]

print(f"X_train : {X_train.shape[0]:,} rows")
print(f"X_test  : {X_test.shape[0]:,} rows\n")

print("Stratification check — class proportions across splits:")
print(f"{'Class':<28} {'Full':>8} {'Train':>8} {'Test':>8}")
print("-" * 56)
for cls in sorted(y_type.unique()):
    full_pct  = (y_type == cls).mean()
    train_pct = (y_train_type == cls).mean()
    test_pct  = (y_test_type == cls).mean()
    print(f"{cls:<28} {full_pct:>7.1%} {train_pct:>7.1%} {test_pct:>7.1%}")

print("\nLabel alignment check:")
print(f"  X_train.index == y_train_type.index    : {(X_train.index == y_train_type.index).all()}")
print(f"  X_train.index == y_train_priority.index: {(X_train.index == y_train_priority.index).all()}")
print(f"  X_test.index  == y_test_type.index     : {(X_test.index == y_test_type.index).all()}")
print(f"  X_test.index  == y_test_priority.index : {(X_test.index == y_test_priority.index).all()}")

## Step 2 — TF-IDF Parameters

Every parameter is justified against our actual data, not copied from tutorials written for different datasets.

| Parameter | Value | Reasoning for our data |
|---|---|---|
| `max_features` | `5000` | Vocabulary after preprocessing is 5,577 unique tokens. Setting 10,000 is nonsensical — the vectorizer would just use all 5,577. Setting 5,000 uses most of the vocabulary while excluding the very lowest-frequency edge cases that contribute no generalizable signal. |
| `ngram_range` | `(1, 2)` | Processed text is already isolated meaningful words. Bigrams like `"network error"`, `"account access"`, `"software update"` carry more information than either word alone. `min_df=3` filters the rare bigrams. |
| `min_df` | `3` | Keeps a term only if it appears in ≥ 3 training documents. With 5,577 unique tokens, many are extremely rare. `min_df=2` is too permissive (noise), `min_df=5` too aggressive. 3 is the middle ground for this dataset size. |
| `max_df` | `0.85` | Ignores any term appearing in > 85% of documents. TF-IDF's built-in stopword mechanism. Any term appearing in 85%+ of tickets is a corpus-wide constant that discriminates nothing. If `"problem"` survived stopword removal and still appears everywhere, `max_df` catches it. |
| `sublinear_tf` | `True` | Tickets average 20.8 words — short text. A word appearing 3 times in a 20-word document is disproportionately powerful compared to 3 times in a 200-word document. `sublinear_tf=True` applies `1 + log(tf)` instead of raw `tf`, preventing any single repeated word from dominating the vector. |

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("=== TF-IDF FIT + TRANSFORM COMPLETE ===\n")
print(f"Actual vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"  (max_features=5000, raw vocab=5,577 — actual may be under 5,000 after min_df/max_df filtering)")
print(f"\nX_train_tfidf shape : {X_train_tfidf.shape}")
print(f"X_test_tfidf  shape : {X_test_tfidf.shape}")

train_density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
test_density  = X_test_tfidf.nnz  / (X_test_tfidf.shape[0]  * X_test_tfidf.shape[1])
print(f"\nMatrix sparsity:")
print(f"  Train density : {train_density:.4f}  ({train_density*100:.2f}% non-zero — expected very sparse)")
print(f"  Test  density : {test_density:.4f}  ({test_density*100:.2f}%)")

## Step 3 — Save Artifacts

| File | Why it must be saved |
|---|---|
| `models/tfidf_vectorizer.pkl` | Carries the vocabulary and IDF weights. Notebook 08 needs it to transform text. Streamlit app needs it at inference time. One of the most critical artifacts in the project. |
| `X_train_tfidf.npz` | Notebook 08 loads this directly — no need to rerun the vectorizer. Sparse `.npz` format preserves sparsity — a dense `.csv` would be ~200× larger. |
| `X_test_tfidf.npz` | Same reason. |
| `y_train_type.csv` / `y_test_type.csv` | Classification labels — used by notebooks 08 and 09. |
| `y_train_priority.csv` / `y_test_priority.csv` | Priority labels — used by notebook 10. |

In [ ]:
# Fitted vectorizer
vectorizer_path = os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl')
joblib.dump(vectorizer, vectorizer_path)
print(f"Saved : {vectorizer_path}  ({os.path.getsize(vectorizer_path)/1024:.1f} KB)")

# Sparse matrices
train_npz_path = os.path.join(PROCESSED_DIR, 'X_train_tfidf.npz')
test_npz_path  = os.path.join(PROCESSED_DIR, 'X_test_tfidf.npz')
scipy.sparse.save_npz(train_npz_path, X_train_tfidf)
scipy.sparse.save_npz(test_npz_path,  X_test_tfidf)
print(f"Saved : {train_npz_path}  ({os.path.getsize(train_npz_path)/1024:.1f} KB)")
print(f"Saved : {test_npz_path}   ({os.path.getsize(test_npz_path)/1024:.1f} KB)")

# Label arrays — save with index=True so row identity is preserved
label_files = {
    'y_train_type'    : y_train_type,
    'y_test_type'     : y_test_type,
    'y_train_priority': y_train_priority,
    'y_test_priority' : y_test_priority,
}
for name, series in label_files.items():
    path = os.path.join(PROCESSED_DIR, f'{name}.csv')
    series.to_csv(path, index=True, header=True)
    print(f"Saved : {path}  ({len(series)} rows)")

print("\n=== ALL ARTIFACTS SAVED ===")

## Step 4 — Vocabulary Diagnostic

Before this notebook is complete, check one dataset-specific issue.

From notebook 04's frequency table, the top 3 tokens in `processed_text` were **`im`**, **`assist`**, **`ive`** — informal contractions that survived NLTK stopword removal.

Two possible outcomes for each:

- **Token IN vocabulary** → check its IDF score. A very low IDF (close to 0) means it appears in nearly every document and will be heavily down-weighted by TF-IDF automatically. Not a problem, but worth noting.
- **Token NOT in vocabulary** → `max_df=0.85` removed it. Ideal outcome — it appeared in 85%+ of tickets equally across all classes and contributes zero discriminative signal.

In [ ]:
vocab      = vectorizer.vocabulary_       # term → column index
idf_scores = vectorizer.idf_               # IDF score per column index

print("=== TEMPLATE ARTIFACT DIAGNOSTIC ===\n")
check_tokens = ['im', 'assist', 'ive']
for token in check_tokens:
    if token in vocab:
        idf = idf_scores[vocab[token]]
        print(f"  '{token}' IN vocabulary   — IDF: {idf:.4f}  (lower = appears in more docs, down-weighted by TF-IDF)")
    else:
        print(f"  '{token}' NOT in vocabulary — max_df filter removed it ✓")

feature_names = vectorizer.get_feature_names_out()
idf_series    = pd.Series(idf_scores, index=feature_names).sort_values()

print("\n=== TOP 15 LOWEST-IDF TERMS (appear in most documents) ===\n")
print(idf_series.head(15).to_string())

print("\n=== TOP 15 HIGHEST-IDF TERMS (rarest terms kept by min_df) ===\n")
print(idf_series.tail(15).to_string())

## Summary

| Step | Action | Key output |
|---|---|---|
| 0 — Verify | Null + empty string check, word count stats, label distributions | All 3 checks pass |
| 1 — Split | 80/20 stratified split `random_state=42`, priority labels aligned via `.loc[index]` | X_train (6,461), X_test (1,616) |
| 2 — Parameters | `max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.85, sublinear_tf=True` | Justified for 5,577-token vocab, 20.8-word avg length |
| 3 — Fit + Transform | `fit_transform(X_train)` then `transform(X_test)` — test data never seen during fitting | Two sparse matrices |
| 4 — Save | Vectorizer `.pkl`, two sparse `.npz`, four label `.csv` files | 7 files |
| 5 — Diagnostic | IDF scores for `im`, `assist`, `ive` — confirm template artifacts are down-weighted or absent | IDF table |

### Loading in downstream notebooks

```python
# Notebook 08 — Classification
import scipy.sparse, joblib, pandas as pd

X_train    = scipy.sparse.load_npz('../data/processed/X_train_tfidf.npz')
X_test     = scipy.sparse.load_npz('../data/processed/X_test_tfidf.npz')
vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')
y_train    = pd.read_csv('../data/processed/y_train_type.csv',     index_col=0).squeeze()
y_test     = pd.read_csv('../data/processed/y_test_type.csv',      index_col=0).squeeze()

# Notebook 10 — Priority Prediction (same matrices, different labels)
y_train_p  = pd.read_csv('../data/processed/y_train_priority.csv', index_col=0).squeeze()
y_test_p   = pd.read_csv('../data/processed/y_test_priority.csv',  index_col=0).squeeze()
```